In [30]:
import pandas as pd
import numpy as np

In [5]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_FILE = RAW_DIR / "era5_land_recovery_2022_2024.nc"

print("Project:", PROJECT_DIR)
print("Output:", RECOVERY_FILE)
print("Directory exists:", RAW_DIR.exists())

Project: /home/trang/Drought_prediction
Output: /home/trang/Drought_prediction/data/raw/era5_land_recovery_2022_2024.nc
Directory exists: True


In [6]:
import cdsapi

dataset = "reanalysis-era5-land-monthly-means"

request = {
    "product_type": ["monthly_averaged_reanalysis_by_hour_of_day"],
    "variable": [
        "potential_evaporation",
        "total_precipitation"
    ],
    "year": ["2022", "2023", "2024"],
    "month": [
        "09", "10", "11", "12",
        "01", "02"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [11.1, 103.3, 8, 107]
}

client = cdsapi.Client()

client.retrieve(
    dataset,
    request
).download(str(RECOVERY_FILE))

2026-11-01 16:17:46,134 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
2026-11-01 16:17:46,761 INFO Request ID is 8131b234-5f90-46ef-bf53-fa8232f67c61
2026-11-01 16:17:46,995 INFO status has been updated to successful


'/home/trang/Drought_prediction/data/raw/era5_land_recovery_2022_2024.nc'

In [7]:
print("File exists:", RECOVERY_FILE.exists())

if RECOVERY_FILE.exists():
    print(f"File size: {RECOVERY_FILE.stat().st_size / 1024**2:.2f} MB")

File exists: True
File size: 0.10 MB


In [8]:
import xarray as xr

try:
    recovery_ds = xr.open_dataset(RECOVERY_FILE)

    print(recovery_ds)
    print("\nVariables:", list(recovery_ds.data_vars))
    print("\nDimensions:", recovery_ds.sizes)
    print("\nTime range:")
    print(recovery_ds.valid_time.min().values)
    print(recovery_ds.valid_time.max().values)

except Exception as e:
    print("Error:", repr(e))

<xarray.Dataset> Size: 176kB
Dimensions:     (valid_time: 18, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 144B 2022-01-02 ... 2024-12-02
    expver      (valid_time) <U4 288B ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    pev         (valid_time, latitude, longitude) float32 88kB ...
    tp          (valid_time, latitude, longitude) float32 88kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-01T09:16 GRIB to CDM+CF via cfgrib-0.9.1...

Variables: ['pev', 'tp']

Dimensions: Frozen({'valid_time': 18, 'latitude': 32, 'lon

In [9]:
for var in ["tp", "pev"]:
    if var in recovery_ds:
        da = recovery_ds[var]

        print(f"\n--- {var} ---")
        print("Units:", da.attrs.get("units"))
        print("Valid values:", int(da.count().values))
        print("Min:", float(da.min().values))
        print("Max:", float(da.max().values))


--- tp ---
Units: m
Valid values: 8892
Min: 2.6926849386654794e-05
Max: 0.02512257918715477

--- pev ---
Units: m
Valid values: 8892
Min: -0.05501305311918259
Max: -0.003484301269054413


In [13]:
for t in recovery_ds["valid_time"].values:
    print(pd.Timestamp(t).strftime("%Y-%m"))

2022-01
2022-02
2022-09
2022-10
2022-11
2022-12
2023-01
2023-02
2023-09
2023-10
2023-11
2023-12
2024-01
2024-02
2024-09
2024-10
2024-11
2024-12


In [14]:
print("Recovery dates:")
print(recovery_ds.valid_time.values)

print("\nVariable attributes:")
for var in ["tp", "pev"]:
    print(f"\n{var}:")
    print(recovery_ds[var].attrs)

Recovery dates:
['2022-01-02T00:00:00.000000000' '2022-02-02T00:00:00.000000000'
 '2022-09-02T00:00:00.000000000' '2022-10-02T00:00:00.000000000'
 '2022-11-02T00:00:00.000000000' '2022-12-02T00:00:00.000000000'
 '2023-01-02T00:00:00.000000000' '2023-02-02T00:00:00.000000000'
 '2023-09-02T00:00:00.000000000' '2023-10-02T00:00:00.000000000'
 '2023-11-02T00:00:00.000000000' '2023-12-02T00:00:00.000000000'
 '2024-01-02T00:00:00.000000000' '2024-02-02T00:00:00.000000000'
 '2024-09-02T00:00:00.000000000' '2024-10-02T00:00:00.000000000'
 '2024-11-02T00:00:00.000000000' '2024-12-02T00:00:00.000000000']

Variable attributes:

tp:
{'GRIB_paramId': np.int64(228), 'GRIB_dataType': 'fc', 'GRIB_numberOfPoints': np.int64(1216), 'GRIB_typeOfLevel': 'surface', 'GRIB_stepUnits': np.int64(1), 'GRIB_gridType': 'regular_ll', 'GRIB_uvRelativeToGrid': np.int64(0), 'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(38), 'GRIB_Ny': np.int64(32), 'GRIB_cfName': 'unknown', 'GRIB_cfVarName': 'tp', 'GRIB_gridDefinitionDe

In [15]:
import cdsapi

dataset = "reanalysis-era5-land-monthly-means"

request_2023_missing = {
    "product_type": ["monthly_averaged_reanalysis_by_hour_of_day"],
    "variable": [
        "potential_evaporation",
        "total_precipitation"
    ],
    "year": ["2023"],
    "month": ["03", "04", "05", "06", "07", "08"],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [11.1, 103.3, 8, 107]
}

MISSING_FILE = RAW_DIR / "era5_land_recovery_2023_03_08.nc"

client = cdsapi.Client()

client.retrieve(
    dataset,
    request_2023_missing
).download(str(MISSING_FILE))

2026-11-01 16:21:21,067 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
2026-11-01 16:21:21,632 INFO Request ID is 5784c983-a490-4be1-bba0-00454dac59ae
2026-11-01 16:21:21,864 INFO status has been updated to accepted
2026-11-01 16:21:36,288 INFO status has been updated to running
2026-11-01 16:21:55,722 INFO status has been updated to successful


'/home/trang/Drought_prediction/data/raw/era5_land_recovery_2023_03_08.nc'

In [16]:
missing_ds = xr.open_dataset(MISSING_FILE)

print(missing_ds)
print([
    pd.Timestamp(t).strftime("%Y-%m")
    for t in missing_ds.valid_time.values
])

<xarray.Dataset> Size: 59kB
Dimensions:     (valid_time: 6, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 48B 2023-03-02 ... 2023-08-02
    expver      (valid_time) <U4 96B ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    pev         (valid_time, latitude, longitude) float32 29kB ...
    tp          (valid_time, latitude, longitude) float32 29kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-01T09:21 GRIB to CDM+CF via cfgrib-0.9.1...
['2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08']


In [18]:
# Ghép hai bộ dữ liệu recovery
recovery_full = xr.concat(
    [recovery_ds, missing_ds],
    dim="valid_time"
).sortby("valid_time")

# Chỉ giữ khoảng thời gian cần phục hồi
recovery_full = recovery_full.sel(
    valid_time=slice("2022-09-01", "2024-02-29")
)

print(recovery_full)

<xarray.Dataset> Size: 176kB
Dimensions:     (valid_time: 18, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 144B 2022-09-02 ... 2024-02-02
    expver      (valid_time) <U4 288B '0001' '0001' '0001' ... '0001' '0001'
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B 0
Data variables:
    pev         (valid_time, latitude, longitude) float32 88kB -0.009677 ... nan
    tp          (valid_time, latitude, longitude) float32 88kB 0.02036 ... nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-01T09:16 GRIB to CDM+CF via cfgrib-0.9.1...


In [19]:
recovery_months = pd.PeriodIndex(
    pd.to_datetime(recovery_full.valid_time.values),
    freq="M"
)

expected_months = pd.period_range(
    start="2022-09",
    end="2024-02",
    freq="M"
)

print("Number of months:", len(recovery_months))
print("Unique months:", recovery_months.is_unique)
print("Missing months:", expected_months.difference(recovery_months))
print("Unexpected months:", recovery_months.difference(expected_months))

assert len(recovery_months) == 18
assert recovery_months.is_unique
assert recovery_months.equals(expected_months)

print("\nRecovery period validated successfully!")

Number of months: 18
Unique months: True
Missing months: PeriodIndex([], dtype='period[M]')
Unexpected months: PeriodIndex([], dtype='period[M]')

Recovery period validated successfully!


In [20]:
recovery_months = pd.PeriodIndex(
    pd.to_datetime(recovery_full.valid_time.values),
    freq="M"
)

expected_months = pd.period_range(
    "2022-09",
    "2024-02",
    freq="M"
)

print("Number of months:", len(recovery_months))
print("Unique months:", recovery_months.is_unique)
print("Missing months:", expected_months.difference(recovery_months).tolist())
print("Unexpected months:", recovery_months.difference(expected_months).tolist())

assert len(recovery_months) == 18
assert recovery_months.is_unique
assert recovery_months.equals(expected_months)

print("Recovery period validated successfully!")

Number of months: 18
Unique months: True
Missing months: []
Unexpected months: []
Recovery period validated successfully!


In [21]:
print("Variables:", list(recovery_full.data_vars))
print("Dimensions:", recovery_full.sizes)

for var in ["tp", "pev"]:
    da = recovery_full[var]
    print(
        f"{var}: units={da.attrs.get('units')}, "
        f"valid={int(da.count().values)}, "
        f"min={float(da.min().values)}, "
        f"max={float(da.max().values)}"
    )

Variables: ['pev', 'tp']
Dimensions: Frozen({'valid_time': 18, 'latitude': 32, 'longitude': 38})
tp: units=m, valid=8892, min=2.1802072296850383e-05, max=0.025188690051436424
pev: units=m, valid=8892, min=-0.05501305311918259, max=-0.003632582724094391


In [23]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

BRONZE_FILE = (
    PROJECT_DIR
    / "data"
    / "bronze"
    / "era5_land"
    / "era5_land_monthly_2000_2025.nc"
)

print("Project:", PROJECT_DIR)
print("Bronze file:", BRONZE_FILE)
print("File exists:", BRONZE_FILE.exists())

Project: /home/trang/Drought_prediction
Bronze file: /home/trang/Drought_prediction/data/bronze/era5_land/era5_land_monthly_2000_2025.nc
File exists: True


In [25]:
bronze_ds = xr.open_dataset(BRONZE_FILE)

print(bronze_ds)

<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 312, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2000-01-01 ... 2025-12-01
    expver      (valid_time) <U4 5kB ...
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 2MB ...
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    swvl1       (valid_time, latitude, longitude) float32 2MB ...
    swvl2       (valid_time, latitude, longitude) float32 2MB ...
    swvl3       (valid_time, latitude, longitude) float32 2MB ...
    swvl4       (valid_time, latitude, longitude) float32 2MB ...
    pev         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_centre:             ecmf
 

In [28]:
print("Bronze:")
print(bronze_period.valid_time.values[:5])

print("\nRecovery:")
print(recovery_full.valid_time.values[:5])

Bronze:
['2022-09-01T00:00:00.000000000' '2022-10-01T00:00:00.000000000'
 '2022-11-01T00:00:00.000000000' '2022-12-01T00:00:00.000000000'
 '2023-01-01T00:00:00.000000000']

Recovery:
['2022-09-02T00:00:00.000000000' '2022-10-02T00:00:00.000000000'
 '2022-11-02T00:00:00.000000000' '2022-12-02T00:00:00.000000000'
 '2023-01-02T00:00:00.000000000']


In [32]:
bronze_period = bronze_ds.sel(
    valid_time=slice("2022-09-01", "2024-02-29")
)

bronze_period = bronze_period.assign_coords(
    month_key=(
        "valid_time",
        pd.to_datetime(bronze_period.valid_time.values).strftime("%Y-%m")
    )
)

recovery_compare = recovery_full.assign_coords(
    month_key=(
        "valid_time",
        pd.to_datetime(recovery_full.valid_time.values).strftime("%Y-%m")
    )
)

bronze_period = bronze_period.swap_dims({"valid_time": "month_key"})
recovery_compare = recovery_compare.swap_dims({"valid_time": "month_key"})

print("Bronze months:", bronze_period.sizes["month_key"])
print("Recovery months:", recovery_compare.sizes["month_key"])
print("Same months:", np.array_equal(
    bronze_period.month_key.values,
    recovery_compare.month_key.values
))

Bronze months: 18
Recovery months: 18
Same months: True


In [33]:
for var in ["tp", "pev"]:
    original = bronze_period[var]
    recovered = recovery_compare[var]

    valid = original.notnull() & recovered.notnull()
    diff = recovered - original

    print(f"\n--- {var} ---")
    print("Original valid:", int(original.count().values))
    print("Recovery valid:", int(recovered.count().values))
    print("Common valid:", int(valid.sum().values))
    print("Changed values:", int(((diff != 0) & valid).sum().values))
    print("Max absolute difference:", float(abs(diff).max().values))
    print("Mean absolute difference:", float(abs(diff).mean().values))


--- tp ---
Original valid: 8892
Recovery valid: 8892
Common valid: 8892
Changed values: 8892
Max absolute difference: 0.012122498825192451
Mean absolute difference: 0.002061365870758891

--- pev ---
Original valid: 8892
Recovery valid: 8892
Common valid: 8892
Changed values: 8892
Max absolute difference: 0.022631250321865082
Mean absolute difference: 0.0020364944357424974


In [35]:
for var in ["tp", "pev"]:
    print(f"\n========== {var} ==========")

    original = bronze_period[var].reset_coords(
        "valid_time", drop=True
    )

    recovered = recovery_compare[var].reset_coords(
        "valid_time", drop=True
    )

    comparison = xr.Dataset({
        "original_mean": original.mean(
            dim=["latitude", "longitude"]
        ),
        "recovery_mean": recovered.mean(
            dim=["latitude", "longitude"]
        ),
        "original_min": original.min(
            dim=["latitude", "longitude"]
        ),
        "recovery_min": recovered.min(
            dim=["latitude", "longitude"]
        ),
        "original_max": original.max(
            dim=["latitude", "longitude"]
        ),
        "recovery_max": recovered.max(
            dim=["latitude", "longitude"]
        ),
    }).to_dataframe().round(6)

    print(comparison)


========== tp ==========
           number expver  original_mean  recovery_mean  original_min  \
month_key                                                              
2022-09         0   0001       0.007561       0.011911      0.003921   
2022-10         0   0001       0.005435       0.009131      0.002605   
2022-11         0   0001       0.005576       0.008730      0.002859   
2022-12         0   0001       0.001296       0.002166      0.000222   
2023-01         0   0001       0.001930       0.003240      0.000966   
2023-02         0   0001       0.000531       0.000888      0.000102   
2023-03         0   0001       0.000167       0.000272      0.000011   
2023-04         0   0001       0.001520       0.002678      0.000444   
2023-05         0   0001       0.004362       0.006476      0.002759   
2023-06         0   0001       0.005222       0.008080      0.002660   
2023-07         0   0001       0.006855       0.010586      0.002991   
2023-08         0   0001       0.00409

In [36]:
for var in ["tp", "pev"]:
    original = bronze_period[var].reset_coords(
        ["valid_time", "number", "expver"],
        drop=True
    )

    recovered = recovery_compare[var].reset_coords(
        ["valid_time", "number", "expver"],
        drop=True
    )

    rows = []

    for month in original.month_key.values:
        o = original.sel(month_key=month).values
        r = recovered.sel(month_key=month).values

        rows.append({
            "month": str(month),
            "bronze_mean": np.nanmean(o),
            "recovery_mean": np.nanmean(r),
            "bronze_min": np.nanmin(o),
            "recovery_min": np.nanmin(r),
            "bronze_max": np.nanmax(o),
            "recovery_max": np.nanmax(r),
        })

    result = pd.DataFrame(rows).round(6)

    print(f"\n========== {var} ==========")
    print(result.to_string(index=False))


========== tp ==========
  month  bronze_mean  recovery_mean  bronze_min  recovery_min  bronze_max  recovery_max
2022-09     0.007561       0.011911    0.003921      0.006655    0.012919      0.024669
2022-10     0.005435       0.009131    0.002605      0.005686    0.009592      0.014432
2022-11     0.005576       0.008730    0.002859      0.005312    0.009643      0.013857
2022-12     0.001296       0.002166    0.000222      0.000406    0.003694      0.007004
2023-01     0.001930       0.003240    0.000966      0.001910    0.003323      0.004736
2023-02     0.000531       0.000888    0.000102      0.000179    0.001779      0.002581
2023-03     0.000167       0.000272    0.000011      0.000022    0.001484      0.002175
2023-04     0.001520       0.002678    0.000444      0.000940    0.004094      0.006511
2023-05     0.004362       0.006476    0.002759      0.004012    0.006643      0.009446
2023-06     0.005222       0.008080    0.002660      0.004573    0.008271      0.015925
2023-0

In [37]:
print("Recovery months:", recovery_full.sizes["valid_time"])

print(
    "Recovery dates:",
    pd.to_datetime(recovery_full.valid_time.values).strftime("%Y-%m").tolist()
)

print("Recovery variables:", list(recovery_full.data_vars))

Recovery months: 18
Recovery dates: ['2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02']
Recovery variables: ['pev', 'tp']


In [38]:
# Kiểm tra tọa độ không gian
assert np.array_equal(
    bronze_ds.latitude.values,
    recovery_full.latitude.values
)

assert np.array_equal(
    bronze_ds.longitude.values,
    recovery_full.longitude.values
)

# Kiểm tra đủ 18 tháng
expected_months = pd.period_range(
    "2022-09", "2024-02", freq="M"
)

recovery_months = pd.PeriodIndex(
    pd.to_datetime(recovery_full.valid_time.values),
    freq="M"
)

assert recovery_months.equals(expected_months)
assert recovery_months.is_unique

# Kiểm tra biến
assert {"tp", "pev"}.issubset(bronze_ds.data_vars)
assert {"tp", "pev"}.issubset(recovery_full.data_vars)

print("All recovery checks passed!")

All recovery checks passed!


In [39]:
# Không thay đổi bronze_ds
recovered_bronze = bronze_ds.copy(deep=True)

# Tạo ánh xạ tháng -> vị trí thời gian trong Bronze
bronze_months = pd.PeriodIndex(
    pd.to_datetime(bronze_ds.valid_time.values),
    freq="M"
)

for recovery_idx, month in enumerate(recovery_months):

    bronze_idx = bronze_months.get_loc(month)

    for var in ["tp", "pev"]:

        recovered_bronze[var].values[bronze_idx, :, :] = (
            recovery_full[var].isel(valid_time=recovery_idx).values
        )

print("Recovery replacement completed!")

Recovery replacement completed!


In [40]:
for var in ["tp", "pev"]:

    print(f"\n--- {var} ---")

    differences = []

    for recovery_idx, month in enumerate(recovery_months):

        bronze_idx = bronze_months.get_loc(month)

        updated = recovered_bronze[var].isel(
            valid_time=bronze_idx
        ).values

        expected = recovery_full[var].isel(
            valid_time=recovery_idx
        ).values

        differences.append(
            np.nanmax(np.abs(updated - expected))
        )

    print("Max difference:", max(differences))


--- tp ---
Max difference: 0.0

--- pev ---
Max difference: 0.0


In [41]:
RECOVERED_DIR = PROJECT_DIR / "data" / "interim" / "era5_land"
RECOVERED_DIR.mkdir(parents=True, exist_ok=True)

RECOVERED_FILE = (
    RECOVERED_DIR
    / "era5_land_monthly_2000_2025_recovered.nc"
)

recovered_bronze.to_netcdf(RECOVERED_FILE)

print("Saved:", RECOVERED_FILE)
print("File exists:", RECOVERED_FILE.exists())

Saved: /home/trang/Drought_prediction/data/interim/era5_land/era5_land_monthly_2000_2025_recovered.nc
File exists: True


In [42]:
# Kiểm tra file đã được tạo
print("File exists:", RECOVERED_FILE.exists())
print("File size (MB):", round(RECOVERED_FILE.stat().st_size / (1024**2), 2))

# Mở lại file từ disk để kiểm tra
recovered_check = xr.open_dataset(RECOVERED_FILE)

print("\nDimensions:")
print(recovered_check.sizes)

print("\nVariables:")
print(list(recovered_check.data_vars))

print("\nTime range:")
print(recovered_check.valid_time.min().values)
print(recovered_check.valid_time.max().values)

File exists: True
File size (MB): 3.2

Dimensions:
Frozen({'valid_time': 312, 'latitude': 32, 'longitude': 38})

Variables:
['d2m', 't2m', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'pev', 'tp']

Time range:
2000-01-01T00:00:00.000000000
2025-12-01T00:00:00.000000000


In [43]:
for var in ["tp", "pev"]:
    da = recovered_check[var]

    values = da.values
    valid = values[np.isfinite(values)]

    print(f"\n===== {var.upper()} =====")
    print("Units:", da.attrs.get("units"))
    print("Valid values:", valid.size)
    print("Missing values:", np.isnan(values).sum())
    print("Min:", float(valid.min()))
    print("Max:", float(valid.max()))
    print("Mean:", float(valid.mean()))
    print("Negative values:", int((valid < 0).sum()))
    print("Zero values:", int((valid == 0).sum()))


===== TP =====
Units: m
Valid values: 154128
Missing values: 225264
Min: 3.140009539492894e-06
Max: 0.0288943350315094
Mean: 0.005688691511750221
Negative values: 0
Zero values: 0

===== PEV =====
Units: m
Valid values: 154128
Missing values: 225264
Min: -0.05750565975904465
Max: -0.0030968785285949707
Mean: -0.007822069339454174
Negative values: 154128
Zero values: 0


In [45]:
from pathlib import Path
import xarray as xr

PROJECT_DIR = Path.cwd().parent

CLIPPED_FILE = (
    PROJECT_DIR
    / "data/silver/era5_land/era5_land_mekong_delta_2000_2025.nc"
)

RECOVERED_FILE = (
    PROJECT_DIR
    / "data/interim/era5_land/era5_land_monthly_2000_2025_recovered.nc"
)

print("Silver file exists:", CLIPPED_FILE.exists())
print("Recovered file exists:", RECOVERED_FILE.exists())

Silver file exists: True
Recovered file exists: True


In [47]:
silver_ds = xr.open_dataset(CLIPPED_FILE).load()
recovered_check = xr.open_dataset(RECOVERED_FILE).load()

print("Silver:", silver_ds.sizes)
print("Recovered:", recovered_check.sizes)

Silver: Frozen({'valid_time': 312, 'latitude': 32, 'longitude': 38})
Recovered: Frozen({'valid_time': 312, 'latitude': 32, 'longitude': 38})


In [48]:
import geopandas as gpd
import regionmask
import numpy as np

BOUNDARY_FILE = (
    PROJECT_DIR
    / "data/silver/boundaries/mekong_delta_boundary.geojson"
)

mekong_boundary = gpd.read_file(BOUNDARY_FILE)

mask = regionmask.mask_geopandas(
    mekong_boundary,
    silver_ds["longitude"],
    silver_ds["latitude"]
)

print("Mask shape:", mask.shape)
print("Inside boundary:", int((mask == 0).sum().values))
print("Outside boundary:", int(mask.isnull().sum().values))

/home/trang/Drought_prediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mask shape: (32, 38)
Inside boundary: 335
Outside boundary: 881


In [49]:
for coord in ["latitude", "longitude", "valid_time"]:
    print(
        coord,
        np.array_equal(
            silver_ds[coord].values,
            recovered_check[coord].values
        )
    )

latitude True
longitude True
valid_time True


In [50]:
# Tạo bản sao, không thay đổi Silver gốc
silver_recovered = silver_ds.copy(deep=True)

# Thay hai biến bằng dữ liệu đã recovery
for var in ["tp", "pev"]:
    silver_recovered[var] = recovered_check[var]

# Giữ lại các ô nằm trong Mekong Delta
silver_recovered = silver_recovered.where(mask == 0)

print(silver_recovered)

<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 312, latitude: 32, longitude: 38)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2kB 2000-01-01 ... 2025-12-01
    expver      (valid_time) <U4 5kB '0001' '0001' '0001' ... '0001' '0001'
  * latitude    (latitude) float64 256B 11.1 11.0 10.9 10.8 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 304B 103.3 103.4 103.5 ... 106.8 106.9 107.0
    number      int64 8B 0
Data variables:
    d2m         (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    t2m         (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    swvl1       (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    swvl2       (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    swvl3       (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    swvl4       (valid_time, latitude, longitude) float32 2MB nan nan ... nan
    pev         (valid_time, latitude, longitude) float32 2MB nan nan ... 

In [51]:
for var in ["tp", "pev"]:
    valid_count = int(silver_recovered[var].notnull().sum().values)

    print(
        var,
        "| valid:", valid_count,
        "| min:", float(silver_recovered[var].min(skipna=True)),
        "| max:", float(silver_recovered[var].max(skipna=True))
    )

tp | valid: 101712 | min: 7.904102858447004e-06 | max: 0.02109147608280182
pev | valid: 101712 | min: -0.05014606565237045 | max: -0.003270551562309265


In [52]:
for var in ["tp", "pev"]:
    expected = recovered_check[var].where(mask == 0)
    actual = silver_recovered[var]

    diff = np.abs(actual.values - expected.values)
    valid_diff = diff[np.isfinite(diff)]

    print(f"\n{var.upper()}")
    print("Max absolute difference:", float(valid_diff.max()))
    print("Mean absolute difference:", float(valid_diff.mean()))
    print("Exactly equal:", np.array_equal(
        actual.values, expected.values, equal_nan=True
    ))


TP
Max absolute difference: 0.0
Mean absolute difference: 0.0
Exactly equal: True

PEV
Max absolute difference: 0.0
Mean absolute difference: 0.0
Exactly equal: True


In [53]:

RECOVERED_SILVER_FILE = (
    PROJECT_DIR
    / "data/silver/era5_land/"
    / "era5_land_mekong_delta_2000_2025_recovered.nc"
)

silver_recovered.to_netcdf(RECOVERED_SILVER_FILE)

print("Saved:", RECOVERED_SILVER_FILE)
print("File exists:", RECOVERED_SILVER_FILE.exists())
print(
    "File size (MB):",
    round(RECOVERED_SILVER_FILE.stat().st_size / (1024**2), 2)
)

Saved: /home/trang/Drought_prediction/data/silver/era5_land/era5_land_mekong_delta_2000_2025_recovered.nc
File exists: True
File size (MB): 11.66


In [54]:
silver_final = xr.open_dataset(RECOVERED_SILVER_FILE)

print("Dimensions:", silver_final.sizes)
print("Variables:", list(silver_final.data_vars))

for var in ["tp", "pev"]:
    print(f"\n{var.upper()}")
    print("Valid:", int(silver_final[var].notnull().sum().values))
    print("Min:", float(silver_final[var].min(skipna=True)))
    print("Max:", float(silver_final[var].max(skipna=True)))
    print("Units:", silver_final[var].attrs.get("units"))

Dimensions: Frozen({'valid_time': 312, 'latitude': 32, 'longitude': 38})
Variables: ['d2m', 't2m', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 'pev', 'tp']

TP
Valid: 101712
Min: 7.904102858447004e-06
Max: 0.02109147608280182
Units: m

PEV
Valid: 101712
Min: -0.05014606565237045
Max: -0.003270551562309265
Units: m


In [55]:
import json

METADATA_FILE = (
    PROJECT_DIR
    / "data/silver/era5_land/metadata.json"
)

if METADATA_FILE.exists():
    with open(METADATA_FILE, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    print(json.dumps(metadata, indent=2, ensure_ascii=False))
else:
    metadata = {}
    print("No existing metadata file.")

{
  "dataset_name": "ERA5-Land Mekong Delta",
  "source_file": "/home/trang/Drought_prediction/data/bronze/era5_land/era5_land_monthly_2000_2025.nc",
  "output_file": "/home/trang/Drought_prediction/data/silver/era5_land/era5_land_mekong_delta_2000_2025.nc",
  "format": "NetCDF",
  "spatial_boundary": "13 provinces/cities in the historical Mekong Delta study boundary",
  "crs": "EPSG:4326",
  "time_range": {
    "start": "2000-01-01",
    "end": "2025-12-01"
  },
  "frequency": "Monthly",
  "dimensions": {
    "valid_time": 312,
    "latitude": 32,
    "longitude": 38
  },
  "variables": [
    "d2m",
    "t2m",
    "swvl1",
    "swvl2",
    "swvl3",
    "swvl4",
    "pev",
    "tp"
  ],
  "valid_grid_cells": 326,
  "missing_data_policy": "Preserve NaN; no imputation",
  "transformations": [
    "Spatial clipping using dissolved Mekong Delta boundary",
    "Preserve original ERA5-Land variable values and units",
    "Retain original grid dimensions"
  ]
}


In [56]:
print("Metadata keys:")
for key, value in metadata.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")
    elif isinstance(value, list):
        print(f"\n{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}: {value}")

Metadata keys:
dataset_name: ERA5-Land Mekong Delta
source_file: /home/trang/Drought_prediction/data/bronze/era5_land/era5_land_monthly_2000_2025.nc
output_file: /home/trang/Drought_prediction/data/silver/era5_land/era5_land_mekong_delta_2000_2025.nc
format: NetCDF
spatial_boundary: 13 provinces/cities in the historical Mekong Delta study boundary
crs: EPSG:4326

time_range:
  start: 2000-01-01
  end: 2025-12-01
frequency: Monthly

dimensions:
  valid_time: 312
  latitude: 32
  longitude: 38

variables:
  - d2m
  - t2m
  - swvl1
  - swvl2
  - swvl3
  - swvl4
  - pev
  - tp
valid_grid_cells: 326
missing_data_policy: Preserve NaN; no imputation

transformations:
  - Spatial clipping using dissolved Mekong Delta boundary
  - Preserve original ERA5-Land variable values and units
  - Retain original grid dimensions


In [57]:
import copy
from datetime import datetime, timezone

recovered_metadata = copy.deepcopy(metadata)

# Cập nhật thông tin file đầu ra
recovered_metadata["output_file"] = str(RECOVERED_SILVER_FILE)

# Ghi nhận lịch sử recovery
recovered_metadata["recovery"] = {
    "status": "completed",
    "source_product": "ERA5-Land monthly averaged reanalysis by hour of day",
    "recovered_variables": ["tp", "pev"],
    "recovery_period": {
        "start": "2022-09",
        "end": "2024-02",
        "months": 18
    },
    "source_files": [
        "era5_land_recovery_2022_2024.nc",
        "era5_land_recovery_2023_03_08.nc"
    ],
    "method": (
        "Replace tp and pev values in the Bronze-derived dataset "
        "with recovered values matched by monthly time and spatial coordinates, "
        "then apply the Mekong Delta spatial mask."
    ),
    "validation": {
        "coordinate_alignment": True,
        "tp_exact_match": True,
        "pev_exact_match": True,
        "max_absolute_difference": 0.0,
        "original_bronze_preserved": True,
        "original_silver_preserved": True
    },
    "units_preserved": {
        "tp": "m",
        "pev": "m"
    },
    "processed_at_utc": datetime.now(timezone.utc).isoformat()
}

# Lưu thành metadata riêng
RECOVERED_METADATA_FILE = (
    PROJECT_DIR
    / "data/silver/era5_land/metadata_recovered.json"
)

with open(RECOVERED_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(recovered_metadata, f, indent=2, ensure_ascii=False)

print("Saved:", RECOVERED_METADATA_FILE)
print("Exists:", RECOVERED_METADATA_FILE.exists())

Saved: /home/trang/Drought_prediction/data/silver/era5_land/metadata_recovered.json
Exists: True


In [58]:
with open(RECOVERED_METADATA_FILE, "r", encoding="utf-8") as f:
    check_metadata = json.load(f)

print(json.dumps(
    check_metadata["recovery"],
    indent=2,
    ensure_ascii=False
))

{
  "status": "completed",
  "source_product": "ERA5-Land monthly averaged reanalysis by hour of day",
  "recovered_variables": [
    "tp",
    "pev"
  ],
  "recovery_period": {
    "start": "2022-09",
    "end": "2024-02",
    "months": 18
  },
  "source_files": [
    "era5_land_recovery_2022_2024.nc",
    "era5_land_recovery_2023_03_08.nc"
  ],
  "method": "Replace tp and pev values in the Bronze-derived dataset with recovered values matched by monthly time and spatial coordinates, then apply the Mekong Delta spatial mask.",
  "validation": {
    "coordinate_alignment": true,
    "tp_exact_match": true,
    "pev_exact_match": true,
    "max_absolute_difference": 0.0,
    "original_bronze_preserved": true,
    "original_silver_preserved": true
  },
  "units_preserved": {
    "tp": "m",
    "pev": "m"
  },
  "processed_at_utc": "2026-11-01T09:36:09.249301+00:00"
}


In [59]:
recovery_quality = pd.DataFrame([
    {
        "variable": "tp",
        "valid_values": int(silver_final["tp"].notnull().sum().values),
        "missing_values": int(silver_final["tp"].isnull().sum().values),
        "min": float(silver_final["tp"].min(skipna=True)),
        "max": float(silver_final["tp"].max(skipna=True)),
        "units": silver_final["tp"].attrs.get("units"),
        "negative_values": int((silver_final["tp"] < 0).sum().values),
        "recovery_applied": True,
        "recovery_period": "2022-09 to 2024-02"
    },
    {
        "variable": "pev",
        "valid_values": int(silver_final["pev"].notnull().sum().values),
        "missing_values": int(silver_final["pev"].isnull().sum().values),
        "min": float(silver_final["pev"].min(skipna=True)),
        "max": float(silver_final["pev"].max(skipna=True)),
        "units": silver_final["pev"].attrs.get("units"),
        "negative_values": int((silver_final["pev"] < 0).sum().values),
        "recovery_applied": True,
        "recovery_period": "2022-09 to 2024-02"
    }
])

RECOVERED_QUALITY_REPORT = (
    PROJECT_DIR
    / "data/silver/era5_land/quality_report_recovered.csv"
)

recovery_quality.to_csv(
    RECOVERED_QUALITY_REPORT,
    index=False
)

print(recovery_quality.to_string(index=False))
print("\nSaved:", RECOVERED_QUALITY_REPORT)

variable  valid_values  missing_values       min       max units  negative_values  recovery_applied    recovery_period
      tp        101712          277680  0.000008  0.021091     m                0              True 2022-09 to 2024-02
     pev        101712          277680 -0.050146 -0.003271     m           101712              True 2022-09 to 2024-02

Saved: /home/trang/Drought_prediction/data/silver/era5_land/quality_report_recovered.csv
